## Problema em Programação Linear

Min: $30 000 X_1 + 35 000 X_2 + 72 000 X_3 + 33  000 X_3$ (em milhares de euros)

S.a:

 $ 12X_1 + 18X_2 + 22X_3 \leq 10 000 $ (peso total de carga para transporte em toneladas)
 
 $ 3000X_1 + 3500X_2 + 5400X_3 \geq 2 100 000 $ (número mínimo de pessoas assistidas)
 
 $ X_3 \geq 30 $ (quantidade mínima de kits premium em centenas)
 
 $ X_1, X_2 \geq 0 $

In [1]:
from pulp import LpMaximize, LpMinimize, LpProblem, LpStatus, lpSum, LpVariable
from pulp import GLPK

model = LpProblem(name="MinCusto", sense=LpMinimize)

# Variáveis de decisão
x = {i: LpVariable(name=f"x{i}", lowBound=0, cat= 'Integer') for i in range(1, 4)}

# Restrições
model += (12 * x[1] + 18* x[2] + 22 * x[3] <= 10000, "carga_total_ton")
model += (3000 * x[1] + 3500 * x[2] + 5400 * x[3] >= 2100000, "pessoas_assistidas")
model += (x[3] >= 30, "kits premium")

# Função objetivo
obj_func = 30000 * x[1] + 35000 * x[2] + 72000 * x[3] + 33000 * x[3]
model += obj_func 

# Formulação (visualização do modelo matemático de forma a conferir os dados)
model

MinCusto:
MINIMIZE
30000*x1 + 35000*x2 + 105000*x3 + 0
SUBJECT TO
carga_total_ton: 12 x1 + 18 x2 + 22 x3 <= 10000

pessoas_assistidas: 3000 x1 + 3500 x2 + 5400 x3 >= 2100000

kits_premium: x3 >= 30

VARIABLES
0 <= x1 Integer
0 <= x2 Integer
0 <= x3 Integer

## Viabilidade das Propostas 

In [2]:
# Custo total
custo_1 = 30000 * 184 + 35000 * 396 + 72000 * 30 + 33000 * 30
custo_2 = 30000 * 646 + 72000 * 30 + 33000 * 30
custo_3 = 30000 * 761 + 35000 * 4 + 72000 * 30 + 33000 * 30
custo_4 = 30000 * 765 + 72000 * 30 + 33000 * 30
print(custo_1,custo_2,custo_3,custo_4)

#Total kits enviados
kits1 = 184 + 396 + 30
kits2 = 646 + 30
kits3 = 761 + 4 + 30
kits4 = 765 + 30
print(kits1, kits2, kits3, kits4)

#Restrição 1
R1_1 = 12 * 184 + 18 * 396 + 22 * 30 
R1_2 = 12 * 646 + 22 * 30 
R1_3 = 12 * 761 + 18 * 4 + 22 * 30
R1_4 = 12 * 765 + 22 * 30
print(R1_1,R1_2,R1_3,R1_4)

#Restrição 2
R2_1 = 3000 * 184 + 3500 * 396 + 5400 * 30
R2_2 = 3000 * 646 + 5400 * 30
R2_3 = 3000 * 761 + 3500 * 4 + 5400 * 30
R2_4 = 3000 * 765 + 5400 * 30
print(R2_1,R2_2,R2_3,R2_4)

22530000 22530000 26120000 26100000
610 676 795 795
9996 8412 9864 9840
2100000 2100000 2459000 2457000


## Estratégias de Otimização no Envio de Kits de Ajuda Humanitária

### Minimização do custo da ajuda humanitária

In [3]:
# Consideram-se três formas de resolver um problema em programação linear:
# 1- status = model.solve() 
# 2- status = model.solve(solver=GLPK(msg=True))
# De modo a obter o relatório de análise de sensibilidade: 
# 3- status = model.solve(GLPK(msg=True, options=['--ranges', 'sensitivity.txt’])) 

# Resolver o Problema
status = model.solve()

# Valor ótimo
print(f"objective: {model.objective.value()}")

# Solução ótima
for var in x.values():
    print(f"{var.name}: {var.value()}")

# Valores das variáveis de desvio
for name, constraint in model.constraints.items():
    print(f"{name}: {constraint.value()}") 

objective: 22530000.0
x1: 646.0
x2: 0.0
x3: 30.0
carga_total_ton: -1588.0
pessoas_assistidas: 0.0
kits_premium: 0.0


### Maximização do total de kits enviados

Max: $X_1 + X_2 + X_3$ (em milhares de euros)

S.a:

 $ 12X_1 + 18X_2 + 22X_3 \leq 10 000 $ (peso total de carga para transporte em toneladas)
 
 $ 3000X_1 + 3500X_2 + 5400X_3 \geq 2 100 000 $ (número mínimo de pessoas assistidas)
 
 $ X_3 \geq 30 $ (quantidade mínima de kits premium em centenas)
 
 $ X_1, X_2 \geq 0 $

In [4]:
model2 = LpProblem(name="MaxKits", sense=LpMaximize)

# Variáveis de decisão
x = {i: LpVariable(name=f"x{i}", lowBound=0, cat= 'Integer') for i in range(1, 4)}

# Restrições
model2 += (12 * x[1] + 18* x[2] + 22 * x[3] <= 10000, "carga_total_ton")
model2 += (3000 * x[1] + 3500 * x[2] + 5400 * x[3] >= 2100000, "pessoas_assistidas")
model2 += (x[3] >= 30, "kits premium")

# Função objetivo
obj_func2 = x[1] + x[2] + x[3]
model2 += obj_func2

# Resolver o Problema
status2 = model2.solve()

# Valor ótimo
print(f"objective: {int(model2.objective.value())}")

# Solução ótima
for var in x.values():
    print(f"{var.name}: {int(var.value())}")

# Valores das variáveis de desvio
for name, constraint in model2.constraints.items():
    print(f"{name}: {int(constraint.value())}") 

objective: 808
x1: 778
x2: 0
x3: 30
carga_total_ton: -4
pessoas_assistidas: 396000
kits_premium: 0


### Abordagem Equilibrada entre Custo e Quantidade

####  Soma Ponderada dos Objetivos Percentuais (SPDP)

In [5]:
import numpy as np
# Criar o modelo
model = LpProblem(name="SPDP", sense=LpMinimize)

# Initializar as variáveis de decisão
x = {i: LpVariable(name=f"x{i}", lowBound=0, cat= 'Integer') for i in range(1, 4)}
dM = {i: LpVariable(name=f"DM{i}", lowBound=0) for i in range(1, 3)}
dm = {i: LpVariable(name=f"Dm{i}", lowBound=0) for i in range(1, 3)}

# Restrições
model += (12 * x[1] + 18 * x[2] + 22 * x[3] <= 10000, "carga_total_ton")
model += (3000 * x[1] + 3500 * x[2] + 5400 * x[3] >= 2100000, "pessoas_assistidas")
model += (x[3] >= 30, "kits premium")
model += (30000 * x[1] + 35000 * x[2] + 72000 * x[3] + 33000 * x[3] - dM[1] + dm[1] == 22500000 , "Meta_custo")
model += (x[1] + x[2] + x[3]  - dM[2] + dm[2] == 808, "Meta_nkits")

# Função objetivo
vam = np.array([22500000,808])
peso = np.array([1,1])
obj_func = ((peso[0]* dM[1] + peso[0]*dm[1])/vam[0]) + ((peso[1]* dM[2] + peso[1]*dm[2])/vam[1])
model += obj_func

# Formulação (visualização do modelo matemático de forma a conferir os dados)
model

SPDP:
MINIMIZE
4.444444444444445e-08*DM1 + 0.0012376237623762376*DM2 + 4.444444444444445e-08*Dm1 + 0.0012376237623762376*Dm2 + 0.0
SUBJECT TO
carga_total_ton: 12 x1 + 18 x2 + 22 x3 <= 10000

pessoas_assistidas: 3000 x1 + 3500 x2 + 5400 x3 >= 2100000

kits_premium: x3 >= 30

Meta_custo: - DM1 + Dm1 + 30000 x1 + 35000 x2 + 105000 x3 = 22500000

Meta_nkits: - DM2 + Dm2 + x1 + x2 + x3 = 808

VARIABLES
DM1 Continuous
DM2 Continuous
Dm1 Continuous
Dm2 Continuous
0 <= x1 Integer
0 <= x2 Integer
0 <= x3 Integer

In [6]:
# Resolver o Problema
status = model.solve()

# Valor ótimo 
print(f"objective: {model.objective.value()}")

# Solução ótima
for var in x.values():
    print(f"{var.name}: {int(var.value())}")
    
# Valores das variáveis de desvio
for name, constraint in model.constraints.items():
    print(f"{name}: {int(constraint.value())}") 


objective: 0.1646996699669967
x1: 646
x2: 0
x3: 30
carga_total_ton: -1588
pessoas_assistidas: 0
kits_premium: 0
Meta_custo: 0
Meta_nkits: 0


#### Objetivo MiniMax

In [7]:
model = LpProblem(name="Problema_MinMax", sense=LpMinimize)

# Initializar as variáveis de decisão
x = {i: LpVariable(name=f"x{i}", lowBound=0, cat= 'Integer') for i in range(1, 4)}
dM = {i: LpVariable(name=f"dM{i}", lowBound=0) for i in range(1, 3)}
dm = {i: LpVariable(name=f"dm{i}", lowBound=0) for i in range(1, 3)}
Q = LpVariable(name=f"Q", lowBound=0)

vam = np.array([22500000,808])
peso = np.array([1,1])
# Restrições
model += (12 * x[1] + 18 * x[2] + 22 * x[3] <= 10000, "carga_total_ton")
model += (3000 * x[1] + 3500 * x[2] + 5400 * x[3] >= 2100000, "pessoas_assistidas")
model += (x[3] >= 30, "kits premium")
model += ((peso[0] * dm[1] * (1/vam[0])) - Q <= 0, "desvio_neg_custo")
model += ((peso[1] * dm[2] * (1/vam[1])) - Q <= 0, "desvio_neg_nkits")
model += ((peso[0] * dM[1] * (1/vam[0])) - Q <= 0, "desvio_pos_custo")
model += ((peso[1] * dM[2] * (1/vam[1])) - Q <= 0, "desvio_pos_nkits")

# Função objetivo
obj_func = Q
model += obj_func

model

Problema_MinMax:
MINIMIZE
1*Q + 0
SUBJECT TO
carga_total_ton: 12 x1 + 18 x2 + 22 x3 <= 10000

pessoas_assistidas: 3000 x1 + 3500 x2 + 5400 x3 >= 2100000

kits_premium: x3 >= 30

desvio_neg_custo: - Q + 4.44444444444e-08 dm1 <= 0

desvio_neg_nkits: - Q + 0.00123762376238 dm2 <= 0

desvio_pos_custo: - Q + 4.44444444444e-08 dM1 <= 0

desvio_pos_nkits: - Q + 0.00123762376238 dM2 <= 0

VARIABLES
Q Continuous
dM1 Continuous
dM2 Continuous
dm1 Continuous
dm2 Continuous
0 <= x1 Integer
0 <= x2 Integer
0 <= x3 Integer

In [8]:
status = model.solve()

# Valor ótimo
model.objective.value()
print(f"objective: {model.objective.value()}")

# Solução ótima
for var in x.values():
    print(f"{var.name}: {var.value()}")
    
print(f"{Q}: {Q.value()}")

# Valores das variáveis de desvio
for name, constraint in model.constraints.items():
    print (f"{name}: {constraint.value()}") 

# Valores de desvio
for var in dm.values():
    print(f"{var.name}: {var.value()}")

# Valores de desvio
for var in dM.values():
    print(f"{var.name}: {var.value()}")

objective: 0.0
x1: 0.0
x2: 0.0
x3: 389.0
Q: 0.0
carga_total_ton: -1442.0
pessoas_assistidas: 600.0
kits_premium: 359.0
desvio_neg_custo: 0.0
desvio_neg_nkits: 0.0
desvio_pos_custo: 0.0
desvio_pos_nkits: 0.0
dm1: 0.0
dm2: 0.0
dM1: 0.0
dM2: 0.0


### Cumprimento do Nível de Aspiração na Distribuição de Kits 

#### Prob1

In [9]:
model = LpProblem(name="Nivel1", sense=LpMinimize)

# variáveis de decisão
x = {i: LpVariable(name=f"x{i}", lowBound=0, cat= 'Integer') for i in range(1, 4)}
dm = LpVariable(name="Dm", lowBound=0)
s = LpVariable(name="s", lowBound=0)

# Adicionar a meta ao modelo
model += (12 * x[1] + 18 * x[2] + 22 * x[3] <= 10000, "carga_total_ton")
model += (3000 * x[1] + 3500 * x[2] + 5400 * x[3] >= 2100000, "pessoas_assistidas")
model += (x[3] >= 30, "kits premium")
model += (x[1] + x[2] + x[3] - dm + s == 808, "Meta_nkits")

# função objetivo
obj_func = dm
model += obj_func

model


Nivel1:
MINIMIZE
1*Dm + 0
SUBJECT TO
carga_total_ton: 12 x1 + 18 x2 + 22 x3 <= 10000

pessoas_assistidas: 3000 x1 + 3500 x2 + 5400 x3 >= 2100000

kits_premium: x3 >= 30

Meta_nkits: - Dm + s + x1 + x2 + x3 = 808

VARIABLES
Dm Continuous
s Continuous
0 <= x1 Integer
0 <= x2 Integer
0 <= x3 Integer

In [10]:
status = model.solve()

# Valor ótimo
model.objective.value()
print(f"objective: {model.objective.value()}")

# Solução ótima
for var in x.values():
    print(f"{var.name}: {var.value()}")
    
print(f"{dm}: {dm.value()}")  
print(f"{s}: {s.value()}")

# Valores das variáveis de desvio
for name, constraint in model.constraints.items():
    print (f"{name}: {constraint.value()}") 


objective: 0.0
x1: 1.0
x2: 1.0
x3: 388.0
Dm: 0.0
s: 418.0
carga_total_ton: -1434.0
pessoas_assistidas: 1700.0
kits_premium: 358.0
Meta_nkits: 0.0


#### Prob2

In [11]:
model = LpProblem(name="Nivel2", sense=LpMinimize)

# variáveis de decisão
x = {i: LpVariable(name=f"x{i}", lowBound=0, cat= 'Integer') for i in range(1, 4)}
dM = LpVariable(name="DM", lowBound=0)
dm = LpVariable(name="Dm", lowBound=0)

# Adicionar a meta ao modelo
model += (12 * x[1] + 18 * x[2] + 22 * x[3] <= 10000, "carga_total_ton")
model += (3000 * x[1] + 3500 * x[2] + 5400 * x[3] >= 2100000, "pessoas_assistidas")
model += (x[3] >= 30, "kits premium")
model += (x[1] + x[2] + x[3] + dm == 808, "Meta_nkits")
model += (dm == 0, "informação_nível prioritário")
model += (30000 * x[1] + 35000 * x[2] + 72000 * x[3] + 33000 * x[3] - dM == 22500000 , "Meta_custo")

# função objetivo
obj_func = dM
model += obj_func

model

Nivel2:
MINIMIZE
1*DM + 0
SUBJECT TO
carga_total_ton: 12 x1 + 18 x2 + 22 x3 <= 10000

pessoas_assistidas: 3000 x1 + 3500 x2 + 5400 x3 >= 2100000

kits_premium: x3 >= 30

Meta_nkits: Dm + x1 + x2 + x3 = 808

informação_nível_prioritário: Dm = 0

Meta_custo: - DM + 30000 x1 + 35000 x2 + 105000 x3 = 22500000

VARIABLES
DM Continuous
Dm Continuous
0 <= x1 Integer
0 <= x2 Integer
0 <= x3 Integer

In [12]:
status = model.solve()

# Valor ótimo
model.objective.value()
print(f"objective: {model.objective.value()}")

# Solução ótima
for var in x.values():
    print(f"{var.name}: {var.value()}")
    
print(f"{dm}: {dm.value()}")  

# Valores das variáveis de desvio
for name, constraint in model.constraints.items():
    print (f"{name}: {constraint.value()}") 


objective: 3990000.0
x1: 778.0
x2: 0.0
x3: 30.0
Dm: 0.0
carga_total_ton: -4.0
pessoas_assistidas: 396000.0
kits_premium: 0.0
Meta_nkits: 0.0
informação_nível_prioritário: 0.0
Meta_custo: 0.0
